# LFAST Exposure Time Calculator 
## Preliminary Version: 2026 Oct 12
## Exposure Time vs Wavelength (400–1800 nm)

This notebook implements the ETC for the LFAST-20X on Kitt Peak, based on Fall 2026 understanding of spectrograph and telescope performance. Telescope throughput analysis follows Choi et al. 2026, SPIE, 14147, https://doi.org/10.1117/12.3103179.

## Per-bin definition (this is the core logic of the whole notebook)

At each wavelength bin $\lambda_j$, with resolution width $\delta\lambda(\lambda_j) = \lambda_j / R_{pow}$:

1. The **signal** for that bin is the stellar electron rate produced *within that resolution element only*
   — the narrow interval $[\lambda_j - \delta\lambda/2,\ \lambda_j + \delta\lambda/2]$ — not the star's
   spectrum as a whole. Each bin is an independent SNR / exposure-time calculation.
2. **$N_*$** is the *total* stellar electron rate falling within that single resolution element — obtained after
   multiplying by $\delta\lambda(\lambda_j)$ and QE, **before** any division by pixel count.
3. **$n_{pix}$** is the number of pixels spanning the resolution elemetn, $w(\lambda_j)$. In this notebook it is
   held constant ($w=3$), even though the code is structured so it could vary
   with $\lambda$ later if needed.
4. **$N_{sky}$** is the per-pixel sky background electron rate at $\lambda_j$, from the SkyCalc online calculator.

## How to use this notebook

- Update the parameters in cell 3 to represent the target scenario you wish to test.

- Choose an instrument (CSHELL or WIYN Bench) and set the appropriate parameters in cell 3.

- The default fiber size in cell 3 is for the 18-micron fibers. This can be changed to a 50-micron fiber (but then each telescope stands alone). If you change the fiber size from 18-microns to 50-microns, change the throughput model in Section 9 to match.

- Generate a target SED model in units of W/sr/m^2/nm and replace the default (Planck) model in cell 5 with your custom model if your source is not well represented by a black body.

- `GRID_MODE` is a switch: `'uniform'` (fast, fixed 1 nm spacing — good for quick runs) or `'resel'`
  (one point per resolution element — slower, but lets you cross-check the uniform-grid result against
  the "true" per-resel resolution, including narrow SkyCalc airglow features that a 1 nm grid smooths over).



## 1. Imports

In [ ]:
import numpy as np
from scipy.integrate import quad
from astropy.io import fits
import matplotlib.pyplot as plt


## 2. Physical constants (SI units)

In [ ]:
H_PLANCK = 6.62607015e-34      # Planck constant, J*s
C_LIGHT = 2.99792458e8         # speed of light, m/s
K_BOLTZMANN = 1.380649e-23     # Boltzmann constant, J/K


## 3. User-adjustable parameters

Change values here and re-run the notebook from this cell down.


In [ ]:
# ---- Wavelength grid settings ----
LAMBDA_MIN_NM = 400.0
LAMBDA_MAX_NM = 1800.0
GRID_MODE = 'resel' #'uniform' # 'uniform' (fast, fixed spacing) or 'resel' (one point per resolution element, slower)
UNIFORM_STEP_NM = 1.0          # only used when GRID_MODE == 'uniform'

# ---- Star parameters ----
T_EFF_K = 5800.0               # stellar effective temperature, K
V_MAG = 15.0                   # apparent V-band magnitude (Vega system)

# ---- V-band calibration reference ----
### Probably does not need to be changed by user except for very specific circumstances.
V_BAND_CENTER_NM = 550.0
V_BAND_WIDTH_NM = 90.0
ZP_V_JY = 3640.0               # Vega-system V-band zero point, Jy (Bessell et al. 1998)

# ---- Target SNR ----
SNR_TARGET = 50.0              # every bin solves for the exposure time needed to reach this SNR, in that resel

# ---- Telescope ----
#Note: Central obscuration of 15.5 cm is already in the throughput budget - don't double count here.
TELESCOPE_DIAMETER_M = 0.726 #0.726 m
TELESCOPE_AREA_M2 = np.pi * (TELESCOPE_DIAMETER_M/2.0)**2
N_TELESCOPES = 20              # 20 telescope on one mount
#Change this if you want to explore what you could do with larger arrays!  20 is the nominal value for the Kitt Peak telescope


#LFAST at Kitt Peak will have two different instruments: CSHELL and the WIYN Bench.  Basic parameters for each are below.
#Choose one to setup your calculation.

# ==== CSHELL (uncomment this OR the Bench) ====
#CSHELL will be available from ~1.08 - 1.75 microns. We expect to have multiple resolutions available:
# - A native resolution of R=18,000 using the input fiber alone. 
# - A resolution of 21,500 using the 200 micron slit, which will come with a throughput loss of XX%. 
# - A resolution of 43,000 using the 100 micron slit, which will come with a throughput loss of YY%
# The CSHELL Free Spectral Range is small: ~ 2.5x10^-3 lambda 
# -- (e.g., at 1.555 microns, a single setup gives 39 Ang of spectrum. 
# -- At 1.08 microns, expect 27 Ang. At 1.75 microns, expect 44 Ang)
# -- Update the wavelength range for your setting above, or leave the default and calculate over the entire LFAST bandpass.

# ---- Spectrograph (these parameters held constant across wavelength) ----
#S_opt = 0.1                    # spectrograph efficiency
#R_POW = 21500.0                # resolving power
#N_PIX = 2.5                    # pixels per resolution element (w)

# ---- Detector ----
#DARK_CURRENT_E_S_PIX = 0.5    # e-/s/pixel
#READ_NOISE_E = 22.0           # e-/pixel, per integration (assuming Fowler sampling with total exposure > 8 seconds)
#QE_CONSTANT = 0.8             # placeholder QE value -> turned into a constant array later, you can
                               # swap in a real QE(lambda) array without changing any other code


# ==== BENCH (uncomment this OR CSHELL) ====
#Note that we cannot support all available configurations of the WIYN Bench. The spectrograph requires
#manual reconfiguration, which cannot be accomplished during the night. We plan to offer 1-2 configurations
#at the beginning of operations, and expand based on user demand. A sample configuration for moderate 
#resolution, centered at visible wavelengths, is listed below. Details of all possible bench configurations are listed on the 
#WIYN website: http://www.wiyn.org/instruments/wiynspecoverview.html   Note that LFAST will use the Bench in 1x1 binning.

# ---- Spectrograph (these parameters held constant across wavelength) ----
S_opt = 0.3
R_POW = 1800                   #(600 lpm grating @ 10.1 deg, lambda_central=5500Ang, delta_lambda=2763 Ang)
N_PIX = 2.2

# ---- Detector ----
DARK_CURRENT_E_S_PIX = 3e-4    # e-/s/pixel
READ_NOISE_E = 3.7             # e-/pixel, per integration (assuming Fowler sampling with total exposure > 8 seconds)
QE_CONSTANT = 0.75             # placeholder QE value -> turned into a constant array later, you can
                               # swap in a real QE(lambda) array without changing any other code


# ---- Fiber ----
FIBER_DIAMETER_ARCSEC = 1.48 #This is the "on-the-sky" diameter of the default 18-micron fiber that goes into the 20-unit combiner.
# We are considering a standalone 50 micron fiber that will not go through the fiber combiner.  
#To test that, change FIBER_DIAMETER_ARCSEC = 4.1  and change N_TELESCOPES = 1 above
OMEGA_FIB_ARCSEC2 = np.pi * (FIBER_DIAMETER_ARCSEC / 2)**2

# ---- Atmospheric extinction on the STELLAR path ----
# Keep this one as 'False', since the throughput model already include the atmospheric extinction term.
APPLY_ATMOSPHERIC_EXTINCTION = False

print(f"Telescope area: {TELESCOPE_AREA_M2:.4f} m^2")
print(f"Fiber solid angle: {OMEGA_FIB_ARCSEC2:.4f} arcsec^2")


## 4. Wavelength grid — the 'uniform' vs 'resel' switch

- **`'uniform'`**: a plain fixed-step grid from 400 to 1800 nm (default step 1 nm). Fast — good for a quick look.
- **`'resel'`**: starts at 400 nm and steps forward by that bin's own resel width
  $\delta\lambda(\lambda_j) = \lambda_j / R_{pow}$ each time, so spacing is finer in the blue and coarser in
  the red. This is slower (tens of thousands of bins) but is the literal per-resel calculation, and is a good cross-check against the uniform-grid result.

Both modes produce a `wavelength_grid` array used by everything downstream, so the rest of the notebook
doesn't care which mode built it.


In [ ]:
def build_uniform_grid(lam_min, lam_max, step):
    return np.arange(lam_min, lam_max + step, step)

def build_resel_grid(lam_min, lam_max, R_pow):
    wavelengths = [lam_min]
    while wavelengths[-1] < lam_max:
        current_wavelength = wavelengths[-1]
        resel_width = current_wavelength / R_pow
        wavelengths.append(current_wavelength + resel_width)
    return np.array(wavelengths)

if GRID_MODE == 'uniform':
    wavelength_grid = build_uniform_grid(LAMBDA_MIN_NM, LAMBDA_MAX_NM, UNIFORM_STEP_NM)
elif GRID_MODE == 'resel':
    wavelength_grid = build_resel_grid(LAMBDA_MIN_NM, LAMBDA_MAX_NM, R_POW)
else:
    raise ValueError("GRID_MODE must be 'uniform' or 'resel'")

# resel width at every bin -- this is delta_lambda(lambda_j) from Eq. 6, needed everywhere downstream
delta_lambda = wavelength_grid / R_POW

print(f"Grid mode: {GRID_MODE}")
print(f"Number of wavelength bins: {len(wavelength_grid)}")


## 5. Stellar model — **model swap point**

`stellar_specific_intensity(wavelength_nm, T_eff)` returns $I_\lambda$ in W sr$^{-1}$ m$^{-2}$ nm$^{-1}$
(Eq. 8). This is currently a blackbody (Planck function) approximation.


In [ ]:
def stellar_specific_intensity(wavelength_nm, T_eff):
    '''
    Specific intensity I_lambda emitted at the stellar surface, in W/sr/m^2/nm.
    Currently: blackbody (Planck function) at T_eff. See markdown above for how to swap this out.
    '''
    wavelength_m = np.asarray(wavelength_nm, dtype=float) * 1e-9
    numerator = 2 * H_PLANCK * C_LIGHT**2
    exponent = H_PLANCK * C_LIGHT / (wavelength_m * K_BOLTZMANN * T_eff)
    denominator = wavelength_m**5 * (np.exp(exponent) - 1)
    intensity_per_meter = numerator / denominator
    intensity_per_nm = intensity_per_meter * 1e-9   # convert W/sr/m^2/m -> W/sr/m^2/nm
    return intensity_per_nm    # unit: W/sr/m^2/nm


## 6. Calibrate the stellar model to the real star 

`alpha` is a single calibration constant for this star (not per-bin) — it gets reused at every wavelength
below to convert the model's surface intensity into flux actually observed at Earth.


In [ ]:
# magnitude -> flux density -> band-integrated flux 
f_nu_V_jy = ZP_V_JY * 10**(-0.4 * V_MAG)
f_nu_V_SI = f_nu_V_jy * 1e-26                                    # Jy -> W/m^2/Hz

delta_nu_V = C_LIGHT * (V_BAND_WIDTH_NM * 1e-9) / (V_BAND_CENTER_NM * 1e-9)**2
F_V = f_nu_V_SI * delta_nu_V                                     # W/m^2, observed flux over the V band

# real integration of the Planck function over the V band (replaces the point approximation)
V_band_lower_nm = V_BAND_CENTER_NM - V_BAND_WIDTH_NM / 2
V_band_upper_nm = V_BAND_CENTER_NM + V_BAND_WIDTH_NM / 2

integral_I_V, integral_error = quad(
    lambda wl: stellar_specific_intensity(wl, T_EFF_K),
    V_band_lower_nm, V_band_upper_nm
)

alpha = F_V / integral_I_V   # stellar angular solid angle / flux-calibration constant, sr

print(f"f_nu_V = {f_nu_V_jy:.4f} Jy")
print(f"F_V = {F_V:.4e} W/m^2")
print(f"Integral of I_lambda over V band = {integral_I_V:.4e} W/m^2/sr  (quad error estimate: {integral_error:.2e})")
print(f"alpha = {alpha:.4e} sr")


## 7. Continuum flux at every bin, with optional spectral-line injection

`F_lambda_continuum` is the pure blackbody continuum at every $\lambda_j$.

`spectral_lines` is empty by default — leave it that way for a pure-continuum run. To inject a line later, add entries like:

```python
spectral_lines = [{'center_nm': 656.3, 'depth': 0.3, 'sigma_nm': 0.05}]
```

`depth` is the fractional flux drop **at the line core** (0.3 = 30% below continuum), and `sigma_nm` controls how narrow/wide the dip is. Multiple lines can be listed;
each is applied independently as a multiplicative dip on top of the continuum.


In [ ]:
F_lambda_continuum = alpha * stellar_specific_intensity(wavelength_grid, T_EFF_K)   # W/m^2/nm, at every bin

spectral_lines = [{'center_nm': 656.3, 'depth': 0.3, 'sigma_nm': 0.05}]
# spectral_lines = [{'center_nm': 656.3, 'depth': 0.3, 'sigma_nm': 0.05}]

def apply_spectral_lines(wl_grid, continuum_flux, line_list):
    '''Multiplies continuum_flux by a Gaussian absorption dip for each line in line_list.'''
    flux_with_lines = continuum_flux.copy()
    for line in line_list:
        center = line['center_nm']
        depth = line['depth']
        sigma = line['sigma_nm']
        dip = depth * np.exp(-0.5 * ((wl_grid - center) / sigma)**2)
        flux_with_lines = flux_with_lines * (1 - dip)
    return flux_with_lines

F_lambda_star = apply_spectral_lines(wavelength_grid, F_lambda_continuum, spectral_lines)


## 8. Load the SkyCalc sky background table

Loads `skytable_secz1.3.fits` (ground level 2640 m, August, airmass = 1.3).
Columns used: `lam` (vacuum wavelength, nm), `flux` (sky radiance, photons s$^{-1}$ m$^{-2}$ µm$^{-1}$
arcsec$^{-2}$), and `trans` (broadband atmospheric transmission — only used if
`APPLY_ATMOSPHERIC_EXTINCTION = True` above).


In [ ]:
SKYCALC_FITS_PATH = 'skytable_secz1.3.fits'   

with fits.open(SKYCALC_FITS_PATH) as hdul:
    sky_lambda_nm = np.array(hdul[1].data['lam'])
    sky_flux_perum = np.array(hdul[1].data['flux'])
    sky_trans = np.array(hdul[1].data['trans'])

# make sure the table is sorted by wavelength (needed for the fast lookups below)
sort_order = np.argsort(sky_lambda_nm)
sky_lambda_nm = sky_lambda_nm[sort_order]
sky_flux_perum = sky_flux_perum[sort_order]
sky_trans = sky_trans[sort_order]

print(f"SkyCalc table loaded: {len(sky_lambda_nm)} rows, {sky_lambda_nm.min():.1f}-{sky_lambda_nm.max():.1f} nm")


## 9. Throughput arrays: $K_{opt}(\lambda)$, $\eta_{fib}(\lambda)$, QE

**This follows the throughput model presented in Choi et al. 2026 SPIE paper on LFAST throughput.**

**By default, this is configured for the 18-micron fiber.  If you are using the 50-micron fiber, change the Kopt and Eta files below to the "50" versions**



In [ ]:
K_opt_wavelength_nm, K_opt_values = np.load('Kopt_wavelength.npy'), np.load('Kopt_value_18.npy')
eta_fib_wavelength_nm, eta_fib_values = np.load('Eta_fib_wavelength.npy'), np.load('Eta_fib_value_18.npy')
# -----------------------------------------------------------------------------------------------------
print(K_opt_wavelength_nm)
print(eta_fib_wavelength_nm)
print(np.mean(K_opt_values))
print(np.mean(eta_fib_values))

In [ ]:
# interpolate each onto the actual computation grid (identity if already the same 1 nm grid)
K_opt_on_grid = np.interp(wavelength_grid, K_opt_wavelength_nm, K_opt_values)
eta_fib_on_grid = np.interp(wavelength_grid, eta_fib_wavelength_nm, eta_fib_values)

# QE as a constant array -- can swap in a real QE(lambda) array later 
QE_values = np.full(len(wavelength_grid), QE_CONSTANT)

# optional atmospheric extinction for the stellar path only (see Section 8 and the parameters cell)
if APPLY_ATMOSPHERIC_EXTINCTION:
    trans_on_grid = np.interp(wavelength_grid, sky_lambda_nm, sky_trans)
else:
    trans_on_grid = np.ones(len(wavelength_grid))

### A note on combining $K_{opt}$ and $\eta_{fib}$

$\eta_{fib}$ (point-source fiber injection loss) was folded into $K_{opt}=0.35$, and the sky calculation "undid" that folding by multiplying by $1/\eta_{fib}$
(since the sky is an extended source that doesn't suffer the point-source coupling loss). Now that
$K_{opt}(\lambda)$ and $\eta_{fib}(\lambda)$ are separate arrays, this notebook applies them directly instead:

- **Star** (point source): `K_opt_on_grid * eta_fib_on_grid` — the injection loss applies.
- **Sky** (extended, fills the fiber uniformly): `K_opt_on_grid` only — no injection loss applied.



## 10. $N_*$ — stellar electron rate per bin

Per the per-bin definitions above: multiply by $\delta\lambda(\lambda_j)$ and QE, and **do not** divide by
$n_{pix}$ — the source is one physical process split across pixels by the optics, not $w$ independent
processes (unlike sky/dark/read noise, which genuinely are independent per pixel — that's why only those
terms carry an $n_{pix}$ factor).


In [ ]:
E_lambda = H_PLANCK * C_LIGHT / (wavelength_grid * 1e-9)     # photon energy at each bin, J
N_lambda_photon_flux = F_lambda_star / E_lambda                # photons/s/m^2/nm

n_p_star = N_lambda_photon_flux * TELESCOPE_AREA_M2 * K_opt_on_grid  * trans_on_grid * S_opt #* eta_fib_on_grid
n_p_star_resel = n_p_star * delta_lambda                       # narrow down to one resel, photons/s
N_star = n_p_star_resel * QE_values * N_TELESCOPES              # e-/s -- this is Howell's N_*, per bin

print(f"N_star range: {N_star.min():.3f} to {N_star.max():.3f} e-/s")


## 11. $N_{sky}$ — sky background per bin

For each bin, average every SkyCalc grid point that falls inside
$[\lambda_j - \delta\lambda/2,\ \lambda_j + \delta\lambda/2]$. If the resel is narrower than
the SkyCalc sampling and no point falls inside, fall back to the single nearest SkyCalc point.

Implementation note: this uses `np.searchsorted` (binary search on the sorted SkyCalc wavelength array) to
find each bin's matching points, rather than scanning the whole 30,000+ row table for every one of the
wavelength bins.


In [ ]:
def find_nearest_sky_value(targets, sky_lambda, sky_flux):
    '''For each wavelength in `targets`, return the sky_flux value at the single closest SkyCalc point.'''
    insert_index = np.searchsorted(sky_lambda, targets)
    insert_index = np.clip(insert_index, 1, len(sky_lambda) - 1)
    before_value = sky_flux[insert_index - 1]
    after_value = sky_flux[insert_index]
    before_wavelength = sky_lambda[insert_index - 1]
    after_wavelength = sky_lambda[insert_index]
    use_before = np.abs(targets - before_wavelength) <= np.abs(after_wavelength - targets)
    return np.where(use_before, before_value, after_value)

def compute_sky_radiance_per_bin(wl_grid, half_widths, sky_lambda, sky_flux):
    '''For each bin, average the SkyCalc points inside it; fall back to the nearest point if none fall inside.'''
    lower_edges = wl_grid - half_widths
    upper_edges = wl_grid + half_widths
    first_index_in_bin = np.searchsorted(sky_lambda, lower_edges, side='left')
    last_index_in_bin = np.searchsorted(sky_lambda, upper_edges, side='right')

    radiance = np.zeros(len(wl_grid))
    has_points_inside = last_index_in_bin > first_index_in_bin

    # bins that captured at least one SkyCalc point: average them
    bins_to_average = np.where(has_points_inside)[0]
    for i in bins_to_average:
        radiance[i] = np.mean(sky_flux[first_index_in_bin[i]:last_index_in_bin[i]])

    # bins that captured zero points: nearest-point fallback
    bins_needing_nearest = np.where(~has_points_inside)[0]
    if len(bins_needing_nearest) > 0:
        radiance[bins_needing_nearest] = find_nearest_sky_value(
            wl_grid[bins_needing_nearest], sky_lambda, sky_flux
        )
    return radiance

sky_radiance_per_bin = compute_sky_radiance_per_bin(
    wavelength_grid, delta_lambda / 2, sky_lambda_nm, sky_flux_perum
)                                                                # photons/s/m^2/micron/arcsec^2

sky_radiance_per_nm = sky_radiance_per_bin / 1000.0             # per micron -> per nm

# propagate through telescope area, fiber solid angle, and optics throughput.
# Sky is an extended source filling the fiber, so no eta_fib injection-loss term here.
n_p_sky = sky_radiance_per_nm * TELESCOPE_AREA_M2 * OMEGA_FIB_ARCSEC2 * K_opt_on_grid * (1/eta_fib_on_grid) * S_opt
n_p_sky_resel = n_p_sky * delta_lambda
n_e_sky_resel = n_p_sky_resel * QE_values
N_sky_per_pixel = n_e_sky_resel * N_TELESCOPES / N_PIX           # e-/s/pixel -- this is Howell's N_sky

print(f"N_sky_per_pixel range: {N_sky_per_pixel.min():.3e} to {N_sky_per_pixel.max():.3f} e-/s/pixel")


## 12. Invert for exposure time 

Solved independently at every bin using $N_*(\lambda_j)$ and $N_{sky}(\lambda_j)$ from above, with
$n_{pix}$, $R_{pow}$, dark current, and read noise held constant.


In [ ]:
# To avoid divide by zero problem: 
eps = 1e-20

N_star = np.maximum(N_star, eps)
N_sky_per_pixel= np.maximum(N_sky_per_pixel, eps)

B = N_PIX * (N_sky_per_pixel + DARK_CURRENT_E_S_PIX)

a_coef = N_star**2
b_coef = -(SNR_TARGET**2) * (N_star + B)
c_coef = -(SNR_TARGET**2) * N_PIX * READ_NOISE_E**2

discriminant = b_coef**2 - 4 * a_coef * c_coef
exposure_time = (-b_coef + np.sqrt(discriminant)) / (2 * a_coef)   # seconds, per bin

print(f"Exposure time range: {exposure_time.min():.2f} to {exposure_time.max():.2f} s")


## 13. Plot: exposure time vs wavelength and calculate representative ET

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(wavelength_grid, exposure_time, lw=0.8)
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Exposure time (s)')
ax.set_yscale('log')
ax.set_ylim(1,10000)
ax.set_title(f'Exposure time vs wavelength (SNR = {SNR_TARGET:.0f} per resel, grid mode = {GRID_MODE})')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()
